# SRA "Lesion" (Synapse Destruction) Experiment

このデモでは、SRAが学習したネットワークが**「機能ごとに完全にモジュール化されている」**ことを証明するための、少しハッカー的な実験を行います。

1. モデルに `copy` と `reverse` をマルチタスク学習させます。
2. `reverse` タスクで頻繁に使われている**特定の専門家（シナプス）の重みをわざとゼロに破壊**します。
3. 破壊後、`reverse` タスクは解けなくなりますが、**`copy` タスクには全く影響が出ない（100%正解し続ける）**ことを確認します。

## 1. 環境セットアップとモデルの準備

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn

sys.path.append('.')

import torch
import torch.nn.functional as F
from src.sra_gpu_models import MoESRAModel
class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

config = MoESRAConfig(
    vocab_size=30,
    d_model=64,
    n_layers=2,
    n_heads=2,
    num_synapses=8,
    k=2,
    max_seq_len=32
)
model = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=128).to(device)
optimizer = make_optimizer(model, lr=0.005)

## 2. マルチタスク学習の実行

In [ ]:
print("Training started... (Copy & Reverse)")
model.train()
epochs = 300
batch_size = 32
tasks = ["copy", "reverse"]

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)
    
    optimizer.zero_grad()
    outputs, routing_weights = model(x)
    
    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = sum(load_balance_loss(rw) for rw in routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

print("Training finished!")

## 3. 破壊前の性能テストと、ターゲットシナプスの特定
それぞれのタスクが解けることを確認し、`reverse`タスクで最も使われているシナプス（ターゲット）を見つけます。

In [ ]:
def test_task(task_name):
    model.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=10, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        outputs, routing_weights = model(x)
        preds = outputs.argmax(dim=-1)
        
    # padding以外で正解している割合
    mask = y != 0 # 0 is PAD
    acc = (preds[mask] == y[mask]).float().mean().item()
    
    # 使われたシナプスの集計 (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.n_synapses)
    
    print(f"[{task_name.upper()}] Accuracy: {acc*100:.1f}%")
    return usage

print("=== 破壊前 (Before Lesion) ===")
copy_usage = test_task("copy")
reverse_usage = test_task("reverse")

# Reverseタスクで最も使われているシナプスを「破壊ターゲット」とする
target_synapse = reverse_usage.argmax().item()
print(f"\n=> Target for destruction: Synapse {target_synapse} (highly active in REVERSE)")

## 4. シナプスの破壊 (Lesion)
ターゲットのシナプスの重みを意図的にゼロに書き換えます。

In [ ]:
layer_to_destroy = 0
print(f"Destroying Synapse {target_synapse} in Layer {layer_to_destroy}...")

with torch.no_grad():
    # MoESRABlockの特定のエキスパートの重みを0にする
    block = model.blocks[layer_to_destroy]
    block.w1.data[target_synapse].zero_()
    block.b1.data[target_synapse].zero_()
    block.w2.data[target_synapse].zero_()
    block.b2.data[target_synapse].zero_()

print("Destruction complete. 💥")

## 5. 破壊後の性能テスト
一部の脳（シナプス）が破壊された状態で、再度テストを行います。
`reverse`の精度は落ちますが、別のシナプスを使っているはずの`copy`は無傷なはずです！

In [ ]:
print("=== 破壊後 (After Lesion) ===")
test_task("copy")
test_task("reverse")

print("\n💡 【考察】")
print("1つの大きなニューラルネット（標準的なTransformer）の一部をゼロで破壊すると、通常は全てのタスクが壊滅します。")
print("しかしSRAでは、タスクごとに独立した専門家経路（シナプス）が形成されているため、")
print("『Reverse用の脳が壊れても、Copy用の脳は無傷で動き続ける』という驚異的な堅牢性（ロバストネス）を示します！")